# 02. MediaPipe Face Detection

**Goal:** Detect faces in real-time and draw bounding boxes.


## 1. What is MediaPipe?
It's a library by Google that gives us pre-trained AI models. We don't train AI; we **use** it.


## 2. Face Mesh vs Face Detection
- **Face Detection**: Fast, just gives a box.
- **Face Mesh**: Slower (slightly), gives 468 points (Mouth, Eyes, Nose).

We will use **Face Mesh** because strict Gaze Tracking requires knowing exactly where the eyes are.


In [ ]:
import cv2
import mediapipe as mp

# Initialize the Mesh model
mp_face_mesh = mp.solutions.face_mesh

# Setup the model with options
face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=3,          # Track up to 3 people
    refine_landmarks=True,    # CRITICAL: Gives us Iris landmarks
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)


## 3. The Detection Loop


In [ ]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret: break
    
    # MediaPipe needs RGB, OpenCV gives BGR
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # INFERENCE (The magic happens here)
    results = face_mesh.process(rgb_frame)
    
    # If faces found...
    if results.multi_face_landmarks:
        count = len(results.multi_face_landmarks)
        cv2.putText(frame, f"Faces: {count}", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
        
        for landmarks in results.multi_face_landmarks:
             # Draw a simple dot on the 'Nose' (Index 1)
             # Landmarks are Normalized (0.0 to 1.0). Convert to pixels.
             h, w, c = frame.shape
             nose = landmarks.landmark[1]
             cx, cy = int(nose.x * w), int(nose.y * h)
             cv2.circle(frame, (cx, cy), 5, (0, 0, 255), -1)

    cv2.imshow('Lesson 02', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


### 🛠️ Debugging
- **Performance**: Note the FPS. MediaPipe is heavy. Ensure you have light!
- **Faces not found**: Detection requires the whole face to be visible.


In [ ]:
import cv2
import mediapipe as mp

mp_face_mesh = mp.solutions.face_mesh
mp_draw = mp.solutions.drawing_utils
mp_styles = mp.solutions.drawing_styles

# Create FaceMesh detector
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,      # video mode
    max_num_faces=3,              # detect up to 3 faces
    refine_landmarks=True,        # includes iris landmarks
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    rgb.flags.writeable = False
    results = face_mesh.process(rgb)
    rgb.flags.writeable = True

    if results.multi_face_landmarks:
        for face_landmarks in results.multi_face_landmarks:
            # Draw ALL landmarks + connections
            mp_draw.draw_landmarks(
                image=frame,
                landmark_list=face_landmarks,
                connections=mp_face_mesh.FACEMESH_TESSELATION,  # dense mesh
                landmark_drawing_spec=None,
                connection_drawing_spec=mp_styles.get_default_face_mesh_tesselation_style()
            )

            # Optional: draw lips / eyes separately (more visible)
            mp_draw.draw_landmarks(
                image=frame,
                landmark_list=face_landmarks,
                connections=mp_face_mesh.FACEMESH_CONTOURS,
                landmark_drawing_spec=None,
                connection_drawing_spec=mp_styles.get_default_face_mesh_contours_style()
            )

    cv2.imshow("Face Mesh Dots", frame)

    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
face_mesh.close()
cv2.destroyAllWindows()


In [ ]:
import cv2
import mediapipe as mp

mp_face_mesh = mp.solutions.face_mesh
mp_draw = mp.solutions.drawing_utils

face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,      # video mode
    max_num_faces=3,              # detect up to 3 faces
    refine_landmarks=True,        # includes iris landmarks
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    rgb.flags.writeable = False
    results = face_mesh.process(rgb)
    rgb.flags.writeable = True

    h, w, _ = frame.shape

    if results.multi_face_landmarks:
        for face_landmarks in results.multi_face_landmarks:
            for lm in face_landmarks.landmark:
                # Convert normalized coords to pixel coords
                x, y = int(lm.x * w), int(lm.y * h)
                # Draw a small dot
                cv2.circle(frame, (x, y), 1, (0, 255, 0), -1)

    if cv2.waitKey(1) == ord('q'):
        break

    cv2.imshow('1', frame)

cap.release()
face_mesh.close()
cv2.destroyAllWindows()